# Teste: Pooling por Produto (3 canais juntos), em vez de por Canal (5 produtos juntos)

Testa o eixo oposto de pooling do que usamos no resto da investigação: em vez de "1 modelo por canal, 5 produtos juntos", treina "1 modelo por produto, 3 canais juntos" -- com `Channel` como feature explícita (one-hot). Checa accuracy (MAPE, one-step) e sinal de preço via SHAP, pra ver se reintroduz o mesmo tipo de confound que já resolvemos, ou se é uma alternativa viável.

> Requisitos: `pip install lightgbm shap pandas numpy openpyxl`


## Libraries, Load Data, Feature Engineering

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap

DATA_PATH = "Business_Case_Data_Set.xlsx"

xl = pd.ExcelFile(DATA_PATH)
ext = xl.parse("Table 1 - External Variables")
actual = xl.parse("Table 2 - Sell In")
raw = ext.merge(actual, on=["Week", "Product", "Channel"], how="inner")
raw["Week"] = pd.to_datetime(raw["Week"])
raw = raw.sort_values(["Product", "Channel", "Week"]).reset_index(drop=True)

TARGET_COL = "Sell_In_Tons"
FEATURE_COLS = [
    "Price_per_kg_USD", "Numeric_Distribution", "Weighted_Distribution",
    "Advertising_Investment_USD", "Promotion_Investment_USD", "Promo_Flag",
    "Lag_1w", "Lag_4w", "Rolling_Mean_4w", "Rolling_Std_4w",
    "Price_Change_Pct", "Month_Sin", "Month_Cos", "Holiday_Flag",
    "Interaction_Price_Promo",
]
HOLDOUT_QUANTILE = 0.85


def engineer_features(df):
    df = df.copy()
    df["Promo_Flag"] = df["Promotion_Type"].notna().astype(int)
    g_target = df.groupby(["Product", "Channel"])[TARGET_COL]
    df["Lag_1w"] = g_target.shift(1)
    df["Lag_4w"] = g_target.shift(4)
    shifted = g_target.shift(1)
    grouped_shifted = shifted.groupby([df["Product"], df["Channel"]])
    df["Rolling_Mean_4w"] = grouped_shifted.rolling(4).mean().reset_index(level=[0, 1], drop=True)
    df["Rolling_Std_4w"] = grouped_shifted.rolling(4).std().reset_index(level=[0, 1], drop=True)
    df["Price_Change_Pct"] = df.groupby(["Product", "Channel"])["Price_per_kg_USD"].pct_change()
    month = df["Week"].dt.month
    df["Month_Sin"] = np.sin(2 * np.pi * month / 12)
    df["Month_Cos"] = np.cos(2 * np.pi * month / 12)
    df["Holiday_Flag"] = 0
    df["Interaction_Price_Promo"] = df["Price_per_kg_USD"] * df["Promo_Flag"]
    return df.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)


def time_split(df):
    cutoff = df.groupby(["Product", "Channel"])["Week"].transform(
        lambda x: x.quantile(HOLDOUT_QUANTILE, interpolation="nearest")
    )
    return df[df["Week"] <= cutoff].copy(), df[df["Week"] > cutoff].copy()


df = engineer_features(raw)

# Channel como feature explicita (one-hot, E-commerce como categoria de referencia)
channel_dummies = pd.get_dummies(df["Channel"], prefix="Channel", drop_first=True).astype(int)
df = pd.concat([df, channel_dummies], axis=1)
FEATURE_COLS_CHANNEL_POOL = FEATURE_COLS + list(channel_dummies.columns)

train, test = time_split(df)
print(f"{len(df)} linhas | Treino: {len(train)} | Teste: {len(test)}")
print(f"Features extras adicionadas: {list(channel_dummies.columns)}")


## Treinar: 1 modelo por PRODUTO, misturando os 3 canais

In [ ]:
BASELINE_PARAMS = dict(
    n_estimators=500, learning_rate=0.1, max_depth=6, num_leaves=31,
    min_child_samples=20, subsample=0.7, colsample_bytree=0.7,
    reg_alpha=1.0, reg_lambda=1.0, min_split_gain=0.0,
    boosting_type="gbdt", objective="regression", metric="mape",
    random_state=42, n_jobs=-1, verbose=-1,
)

pooled_by_product_models = {}
for product, sub in train.groupby("Product"):
    X, y = sub[FEATURE_COLS_CHANNEL_POOL], sub[TARGET_COL]
    model = lgb.LGBMRegressor(**BASELINE_PARAMS)
    model.fit(X, y)
    pooled_by_product_models[product] = model

print(f"{len(pooled_by_product_models)} modelos treinados (1 por produto, 3 canais juntos).")


## Accuracy (MAPE, one-step) -- geral por produto, e detalhado por produto-canal

In [ ]:
def evaluate_model(y_true, y_pred):
    mape = float(np.mean(np.abs(y_true - y_pred) / y_true))
    return {"MAPE": mape, "Accuracy": 1 - mape}


rows = []
for product, model in pooled_by_product_models.items():
    for channel in test["Channel"].unique():
        sub = test[(test.Product == product) & (test.Channel == channel)]
        if len(sub) == 0:
            continue
        X, y_true = sub[FEATURE_COLS_CHANNEL_POOL], sub[TARGET_COL].values
        y_pred = model.predict(X)
        rows.append({"Product": product, "Channel": channel, "n_test": len(sub), **evaluate_model(y_true, y_pred)})

acc_by_product_channel = pd.DataFrame(rows)

acc_by_product = acc_by_product_channel.groupby("Product").apply(
    lambda g: pd.Series({"Accuracy": 1 - np.average(g["MAPE"], weights=g["n_test"])})
).reset_index()

print("=== Accuracy por PRODUTO (todos os canais juntos) ===")
print(acc_by_product.to_string(index=False))
print()
print("=== Accuracy detalhada por Produto x Canal ===")
print(acc_by_product_channel[["Product", "Channel", "n_test", "Accuracy"]].to_string(index=False))


## Sinal de preço via SHAP -- por produto, e quebrado por canal dentro do produto

O teste central: será que misturar canais no mesmo modelo reintroduz confusão de sinal, do mesmo jeito que misturar produtos introduzia antes?

In [ ]:
sign_rows = []
for product, model in pooled_by_product_models.items():
    explainer = shap.TreeExplainer(model)
    price_idx = FEATURE_COLS_CHANNEL_POOL.index("Price_per_kg_USD")

    # Sinal GERAL do produto (todos os canais misturados)
    X_all = df[df.Product == product][FEATURE_COLS_CHANNEL_POOL]
    shap_all = explainer.shap_values(X_all)
    pct_positive_all = float((shap_all[:, price_idx] > 0).mean() * 100)
    sign_rows.append({"Product": product, "Channel": "TODOS (pooled)", "Pct_sinal_positivo": pct_positive_all})

    # Sinal QUEBRADO por canal, dentro do mesmo modelo pooled
    for channel in df["Channel"].unique():
        X_ch = df[(df.Product == product) & (df.Channel == channel)][FEATURE_COLS_CHANNEL_POOL]
        if len(X_ch) == 0:
            continue
        shap_ch = explainer.shap_values(X_ch)
        pct_positive_ch = float((shap_ch[:, price_idx] > 0).mean() * 100)
        sign_rows.append({"Product": product, "Channel": channel, "Pct_sinal_positivo": pct_positive_ch})

sign_table = pd.DataFrame(sign_rows)
print(sign_table.to_string(index=False))
print()
print("-> Alvo ideal ~50% (nem 0% nem 100% -- ver discussao sobre SHAP vs PDP no notebook de Experimentos).")
print("   Se os 3 canais de um produto tiverem % bem diferentes entre si, e diferentes do")
print("   'TODOS (pooled)', isso sugere confound de canal, no mesmo espirito do confound")
print("   de produto que ja identificamos antes.")
